# EyeCU 4.0 — Football Detector Training (Google Colab)

Trains and compares football-specific detectors on the EyeCU dataset built by
`tools/build_dataset.py`.

**Classes:** `0 player`, `1 goalkeeper`, `2 referee`, `3 ball` — team identity stays in
`trackers/team_assigner.py`, it is *not* a detector class.

**Before running:** `Runtime → Change runtime type → GPU` (T4 is enough; A100/L4 is faster).

**Experiments** (TODO.md, Phase 3):

| # | Model | imgsz | Purpose |
|---|---|---|---|
| A | YOLOv8s | 640 | cheap compatibility baseline |
| B | YOLO26n | 960 | fastest viable local model |
| C | YOLO26s | 960 | **primary candidate** |
| D | YOLO26s | 1280 | small-player / ball recall |


## 1. Environment

In [ ]:
!pip install -q -U ultralytics

import torch, ultralytics
from ultralytics import YOLO

print('ultralytics', ultralytics.__version__)
print('torch      ', torch.__version__)
print('cuda       ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu        ', torch.cuda.get_device_name(0))
else:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> GPU, then rerun.')

## 2. Dataset

Upload `football_dataset.zip` (from `python tools/build_dataset.py --zip`) to your Drive,
then set `ZIP_PATH` below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH  = '/content/drive/MyDrive/EyeCU/football_dataset.zip'  # <-- edit me
DRIVE_OUT = '/content/drive/MyDrive/EyeCU/runs'                  # trained weights land here
DATA_ROOT = '/content/football_dataset'

In [ ]:
import os, shutil, yaml
from pathlib import Path

assert os.path.exists(ZIP_PATH), f'Not found: {ZIP_PATH}'

if os.path.exists(DATA_ROOT):
    shutil.rmtree(DATA_ROOT)
shutil.unpack_archive(ZIP_PATH, DATA_ROOT)

# build_dataset.py wrote an absolute Windows path; point it at the Colab copy.
YAML_PATH = f'{DATA_ROOT}/football.yaml'
cfg = yaml.safe_load(open(YAML_PATH))
cfg['path'] = DATA_ROOT
yaml.safe_dump(cfg, open(YAML_PATH, 'w'), sort_keys=False)
print(cfg)

In [ ]:
# Sanity check: counts per split, and no image shared between splits.
from collections import Counter

NAMES = [cfg['names'][i] for i in sorted(cfg['names'])]
stems = {}
for split in ('train', 'val', 'test'):
    img_dir = Path(DATA_ROOT) / split / 'images'
    lbl_dir = Path(DATA_ROOT) / split / 'labels'
    imgs = sorted(img_dir.glob('*.jpg')) + sorted(img_dir.glob('*.png'))
    stems[split] = {p.stem for p in imgs}
    counts, empty = Counter(), 0
    for p in imgs:
        txt = lbl_dir / f'{p.stem}.txt'
        lines = [l for l in txt.read_text().splitlines() if l.strip()] if txt.exists() else []
        if not lines:
            empty += 1
        counts.update(NAMES[int(l.split()[0])] for l in lines)
    print(f'{split:<6} {len(imgs):>5} images, {empty} empty  ' +
          '  '.join(f'{n}={counts.get(n, 0)}' for n in NAMES))

for a, b in (('train', 'val'), ('train', 'test'), ('val', 'test')):
    overlap = stems[a] & stems[b]
    assert not overlap, f'LEAK: {len(overlap)} images shared between {a} and {b}'
print('\nNo image appears in more than one split.')

## 3. Training helper

Run a short pilot first (`epochs=10`) to measure VRAM and per-epoch time, then extrapolate
before committing to a long run.

In [ ]:
import time, json

RESULTS = {}

def train(name, weights, imgsz=960, epochs=100, patience=20, batch=-1, **kw):
    """Train one experiment; returns val metrics. batch=-1 lets Ultralytics auto-size."""
    print(f'\n=== {name}: {weights} @ {imgsz}px, {epochs} epochs ===')
    try:
        model = YOLO(weights)
    except Exception as e:
        print(f'!! {weights} unavailable in ultralytics {ultralytics.__version__}: {e}')
        print('   Skipping. Use a YOLO11/YOLOv8 equivalent instead.')
        return None

    t0 = time.time()
    model.train(data=YAML_PATH, epochs=epochs, imgsz=imgsz, batch=batch,
                patience=patience, project='/content/runs', name=name,
                exist_ok=True, seed=0, **kw)
    mins = (time.time() - t0) / 60

    m = model.val(data=YAML_PATH, imgsz=imgsz, split='val')
    per_class = {NAMES[int(c)]: float(m.box.maps[i])
                 for i, c in enumerate(m.box.ap_class_index)}
    RESULTS[name] = {
        'weights': weights, 'imgsz': imgsz, 'epochs': epochs,
        'train_minutes': round(mins, 1),
        'mAP50': float(m.box.map50), 'mAP50_95': float(m.box.map),
        'precision': float(m.box.mp), 'recall': float(m.box.mr),
        'per_class_mAP50_95': per_class,
        'best_pt': f'/content/runs/{name}/weights/best.pt',
    }
    print(json.dumps(RESULTS[name], indent=2))
    return RESULTS[name]

In [ ]:
# PILOT — 10 epochs to confirm the dataset trains and to time a full run.
train('pilot_yolov8s_640', 'yolov8s.pt', imgsz=640, epochs=10, patience=0)

## 4. Experiments A–D

Colab disconnects on idle — run these one cell at a time and keep the tab open.

In [ ]:
# A — compatibility baseline
train('A_yolov8s_640', 'yolov8s.pt', imgsz=640, epochs=50)

In [ ]:
# B — fastest viable local model
train('B_yolo26n_960', 'yolo26n.pt', imgsz=960, epochs=100)

In [ ]:
# C — primary candidate
train('C_yolo26s_960', 'yolo26s.pt', imgsz=960, epochs=100)

In [ ]:
# D — higher resolution for small players and the ball.
# If this OOMs, set batch to a fixed small number (e.g. batch=4).
train('D_yolo26s_1280', 'yolo26s.pt', imgsz=1280, epochs=100)

## 5. Compare, then score the winner on the held-out test matches

In [ ]:
import pandas as pd

df = pd.DataFrame(RESULTS).T
cols = ['imgsz', 'mAP50', 'mAP50_95', 'precision', 'recall', 'train_minutes']
display(df[cols].sort_values('mAP50_95', ascending=False))

display(pd.DataFrame({k: v['per_class_mAP50_95'] for k, v in RESULTS.items()}).T)

In [ ]:
# The test split holds matches never seen in training or validation.
# Score it ONCE, on the model already chosen from val — otherwise it stops being held out.
BEST = max(RESULTS, key=lambda k: RESULTS[k]['mAP50_95'])
print('Selected on val:', BEST)

best_model = YOLO(RESULTS[BEST]['best_pt'])
tm = best_model.val(data=YAML_PATH, imgsz=RESULTS[BEST]['imgsz'], split='test')

print(f'\nTEST mAP50 {tm.box.map50:.4f}  mAP50-95 {tm.box.map:.4f}')
for i, c in enumerate(tm.box.ap_class_index):
    print(f'  {NAMES[int(c)]:<11} mAP50-95={tm.box.maps[i]:.4f}')

## 6. Export and save to Drive

In [ ]:
onnx_path = best_model.export(format='onnx', imgsz=RESULTS[BEST]['imgsz'], simplify=True)

os.makedirs(DRIVE_OUT, exist_ok=True)
for src in (RESULTS[BEST]['best_pt'], str(onnx_path)):
    dst = os.path.join(DRIVE_OUT, f'{BEST}_' + os.path.basename(src))
    shutil.copy2(src, dst)
    print('saved', dst)

results_path = os.path.join(DRIVE_OUT, 'experiments.json')
json.dump({'selected': BEST,
           'test_mAP50': float(tm.box.map50),
           'test_mAP50_95': float(tm.box.map),
           'experiments': RESULTS},
          open(results_path, 'w'), indent=2)
print('saved', results_path)

## 7. Back in the repo

1. Download `best.pt` from Drive into the repo root as e.g. `eyecu_football.pt`.
2. Run the pipeline against it — no Roboflow, no network:

   ```bash
   python run_pipeline.py --input input-videos/short.mp4 \
       --yolo-model eyecu_football.pt --max-frames 300
   ```
3. Compare FPS and detection quality against the Roboflow baseline in `RESULTS.md`.
4. Then continue with TODO.md Phase 4 (detection post-processing) and
   Phase 5 (ByteTrack vs BoT-SORT on *frozen* detections).

The detector must be frozen before trackers are compared — otherwise you cannot tell
which change caused which result.